# OmniCT — Colab training notebook

Runs the full OmniCT experiment grid (4 methods × 3 seeds + saliency) on a Google Colab A100 in roughly 4–6 hours sequential wall clock.

**Before you start**:
1. Set runtime to A100 GPU (`Runtime → Change runtime type → A100 GPU`).
2. Make sure your repo is pushed to GitHub. Edit `GITHUB_REPO` below if your URL is different.
3. Run cells top-to-bottom. Each section is idempotent.

## 1. Hardware & runtime sanity check

In [ ]:
!nvidia-smi | head -20
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Torch:', torch.__version__)

## 2. Mount Google Drive (for persistent results)

Colab disks are wiped when the runtime ends. We back up `results/`, `report/figures/`, and `data/manifests/` to Drive so a disconnect doesn't lose work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/OmniCT'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive backup root:', DRIVE_ROOT)

## 3. Clone repo & install dependencies

Colab already ships with `torch`, `numpy`, `pandas`, `scikit-learn`, `matplotlib`, `pyyaml`, `tqdm`, `tensorboard`, etc. We only install what's missing.

In [ ]:
GITHUB_REPO = 'https://github.com/pranavraghav75/OmniCT.git'  # ← EDIT if different
BRANCH = 'main'

import os, subprocess
if not os.path.isdir('/content/OmniCT/.git'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, GITHUB_REPO, '/content/OmniCT'], check=True)
else:
    subprocess.run(['git', '-C', '/content/OmniCT', 'pull'], check=True)
%cd /content/OmniCT
!git log -1 --oneline

In [ ]:
%pip install -q monai==1.3.2 nibabel pydicom SimpleITK omegaconf einops peft huggingface_hub timm
%pip install -q seaborn
print('Done installing.')

In [ ]:
import sys
if '/content/OmniCT' not in sys.path:
    sys.path.insert(0, '/content/OmniCT')
import torch, monai, sklearn, numpy, pandas, omegaconf
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('monai', monai.__version__)
print('sklearn', sklearn.__version__, 'numpy', numpy.__version__)

## 4. Smoke test on synthetic data (≤2 minutes)

Confirms training pipeline + GPU work end-to-end before we touch real data.

In [ ]:
!python -m src.training.train \
  --config src/configs/baseline_cnn3d.yaml \
  --override seed=0 device=cuda train.epochs=1 \
    data.synthetic_n_train=32 data.synthetic_n_val=8 data.synthetic_n_test=8 \
    data.spatial_size=[64,64,64] train.batch_size=4 train.num_workers=2 \
  --run_name colab_smoke_cnn3d

In [ ]:
import json, pathlib
metrics = json.loads(pathlib.Path('results/colab_smoke_cnn3d/metrics.json').read_text())
print(json.dumps(metrics, indent=2))

## 5. Download LUNA25 subset (200 balanced volumes, ~5–10 GB)

Set `MAX_VOLUMES` lower (e.g. 100) if you want faster downloads / shorter runs.

If you have a HuggingFace token, set it with `os.environ['HF_TOKEN'] = '...'` before running.

In [ ]:
MAX_VOLUMES = 200  # adjust if you're tight on time
!python data/download_flare.py --out data/raw --max_volumes {MAX_VOLUMES} --balance --seed 0

In [ ]:
!python -m src.data.make_splits \
  --labels data/manifests/labels.csv \
  --out_dir data/manifests/splits \
  --train 0.7 --val 0.15 --test 0.15 --seed 0
!ls data/manifests/splits/
!head -5 data/manifests/labels.csv

## 6. Helper: run a config across N seeds

In [ ]:
import subprocess, time, pathlib, shlex

def run_grid(config_path, n_seeds=3, extra_overrides=None, real_data=True):
    extras = list(extra_overrides or [])
    if real_data:
        extras.append('data.synthetic=false')
    cfg_stem = pathlib.Path(config_path).stem
    for seed in range(n_seeds):
        run_name = f'{cfg_stem}_seed{seed}'
        cmd = ['python', '-m', 'src.training.train',
               '--config', config_path,
               '--run_name', run_name,
               '--override', f'seed={seed}', 'device=cuda', *extras]
        print('\n>>>', ' '.join(shlex.quote(c) for c in cmd))
        t0 = time.time()
        subprocess.run(cmd, check=False)
        print(f'    → finished in {(time.time()-t0)/60:.1f} min')

def backup_to_drive():
    """Copy results+figures+manifests to Drive so disconnects are recoverable."""
    !mkdir -p {DRIVE_ROOT}/results {DRIVE_ROOT}/report_figures {DRIVE_ROOT}/manifests
    !rsync -a --delete results/ {DRIVE_ROOT}/results/
    !rsync -a --delete report/figures/ {DRIVE_ROOT}/report_figures/ 2>/dev/null || true
    !rsync -a --delete data/manifests/ {DRIVE_ROOT}/manifests/
    print('Backed up to', DRIVE_ROOT)

## 7. Baseline 1: class-prior (instant)

In [ ]:
run_grid('src/configs/baseline_prior.yaml', n_seeds=3,
         extra_overrides=['train.epochs=1', 'train.batch_size=4', 'train.num_workers=2'])
backup_to_drive()

## 8. Baseline 2: small 3D CNN (~1–1.5 hr total)

In [ ]:
run_grid('src/configs/baseline_cnn3d.yaml', n_seeds=3,
         extra_overrides=['train.epochs=20', 'train.batch_size=4', 'train.num_workers=2',
                          'data.spatial_size=[96,96,96]'])
backup_to_drive()

## 9. Linear probe on frozen backbone (~30–60 min)

Edit `src/configs/linear_probe.yaml` if you want a real foundation model (3DINO/SPECTRE) vs. `random_init`.

In [ ]:
run_grid('src/configs/linear_probe.yaml', n_seeds=3,
         extra_overrides=['train.epochs=15', 'train.batch_size=4', 'train.num_workers=2',
                          'data.spatial_size=[96,96,96]'])
backup_to_drive()

## 10. LoRA fine-tune (~2–3 hr total)

In [ ]:
run_grid('src/configs/lora.yaml', n_seeds=3,
         extra_overrides=['train.epochs=20', 'train.batch_size=4', 'train.num_workers=2',
                          'data.spatial_size=[96,96,96]'])
backup_to_drive()

## 11. Aggregate results into mean±std table

In [ ]:
!python -m src.training.aggregate_results \
  --runs_dir results --out results/tables/main.csv --split test
import pandas as pd
tbl = pd.read_csv('results/tables/main.csv')
tbl

## 12. Saliency / Grad-CAM panel

In [ ]:
import os
os.makedirs('report/figures', exist_ok=True)
!python -m src.explain.run_saliency \
  --config src/configs/lora.yaml \
  --checkpoint results/lora_seed0/best.pt \
  --override data.synthetic=false device=cuda \
  --n_samples 6 \
  --methods grad_cam grad_x_input \
  --out report/figures/saliency_panel.pdf

In [ ]:
backup_to_drive()
print('All done. Inspect Drive at:', DRIVE_ROOT)

## Optional: data-efficiency experiment (subsample train set)

Drop in if you have time. Trains LoRA at 25%, 50%, 100% of training data.

In [ ]:
# Quick data-efficiency hack: subsample the train split file in-place,
# train, restore. Only run if you've already finished the main grid.
import shutil, random, pathlib

train_full = pathlib.Path('data/manifests/splits/train.txt')
train_bak  = train_full.with_suffix('.full.txt')
if not train_bak.exists():
    shutil.copy(train_full, train_bak)
full_lines = train_bak.read_text().strip().splitlines()
print('full train size:', len(full_lines))

for frac in [0.25, 0.5, 1.0]:
    for seed in range(3):
        rng = random.Random(seed)
        sub = rng.sample(full_lines, max(2, int(len(full_lines) * frac)))
        train_full.write_text('\n'.join(sub) + '\n')
        run_name = f'lora_frac{int(frac*100)}_seed{seed}'
        !python -m src.training.train \
          --config src/configs/lora.yaml \
          --run_name {run_name} \
          --override seed={seed} device=cuda data.synthetic=false \
            train.epochs=20 train.batch_size=4 train.num_workers=2

shutil.copy(train_bak, train_full)
backup_to_drive()